In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split

In [ ]:
def ndcg_at_k(y_true, y_pred, k=10):
    """nDCG@Kを計算する関数"""
    def dcg(scores):
        return np.sum((2**scores - 1) / np.log2(np.arange(2, scores.size + 2)))
    
    order = np.argsort(y_pred)[::-1][:k]
    ideal_order = np.argsort(y_true)[::-1][:k]
    
    return dcg(y_true[order]) / (dcg(y_true[ideal_order]) + 1e-10)

In [ ]:
# XGBoost用カスタム評価関数
def ndcg_eval(preds, dtrain):
    labels = dtrain.get_label()
    return 'ndcg', ndcg_at_k(labels, preds, k=22)

In [ ]:
train_df = pd.read_csv('../001.preprocess/train/train_A.csv')
test_df = pd.read_csv('../001.preprocess/test/test_A.csv')

In [ ]:
#人気順の最大値を31にする
train_df["rank"] = train_df["rank"].clip(upper=31)

In [ ]:
# 特徴量とターゲットの設定
X = train_df[["user_id", "product_id"]]
y = train_df["rank"]

In [ ]:
# データの分割
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# XGBoost用データフォーマットに変換
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

In [ ]:
# XGBoostのモデル学習
dparams = {
    "objective": "rank:pairwise",
    "eval_metric": "ndcg",
    "ndcg_at_k": 22,
    "tree_method": "hist",
    "max_depth": 6,
    "learning_rate": 0.1,
    "lambda": 1.0,
    "ndcg_exp_gain": False
}

In [ ]:
# モデルの学習
bst = xgb.train(dparams, dtrain, num_boost_round=100, evals=[(dvalid, "valid")], early_stopping_rounds=10, feval=ndcg_eval)

In [ ]:
# 評価データの予測
all_product_ids = train_df["product_id"].unique()
predictions = []
    
for user in test_df["user_id"].unique():
    user_data = pd.DataFrame({"user_id": [user] * len(all_product_ids), "product_id": all_product_ids})
    dtest = xgb.DMatrix(user_data)
    user_preds = bst.predict(dtest)
        
    user_pred_df = pd.DataFrame({"user_id": user, "product_id": all_product_ids, "pred_score": user_preds})
    user_pred_df.sort_values("pred_score", ascending=False, inplace=True)
    user_pred_df["rank"] = range(len(user_pred_df))
    predictions.append(user_pred_df)
 
test_df = pd.concat(predictions, ignore_index=True)

In [ ]:
# nDCGを計算
ndcg_scores = []
for user in test_df["user_id"].unique():
    true_scores = train_df.loc[train_df["user_id"] == user, "score"].values
    pred_scores = test_df.loc[test_df["user_id"] == user, "pred_score"].values
    ndcg_scores.append(ndcg_at_k(true_scores, pred_scores, k=10))

print(f"Mean nDCG@10: {np.mean(ndcg_scores):.4f}")

: 